# 03_Level2_Classifier

Predicts each HBCA cell's epithelial cell state from its expression of the cell state marker genes.

- **Model:** multinomial logistic regression with a **group elastic net** penalty. Each gene's coefficients across all classes form one group: the lasso part drops whole genes (for every class at once), and the ridge part spreads weight across correlated genes.
- **Classes:** the 10 epithelial cell states (`LASP1`–`LASP5`, `LHS1`–`LHS3`, `BMYO1`–`BMYO2`). Every other cell is labelled `"Other"`.
- **Features:** the log-normalized expression of the cell state marker genes, scaled to unit variance, plus an intercept column (which is not penalized).
- **Tuning:** 70/30 train/test split, then 5-fold cross-validation on the training set over the penalty strength (`alpha`) and the lasso/ridge mix (`l1_ratio`), scored by held-out log-loss.

**Needs** (from `02_Marker_Gene_Matrices.R`, in `<STORAGE_DIR>/<run>/`): `reed_state_marker_expr.mtx.gz`, `.genes.txt.gz`, `.cells.txt.gz`, and `reed_cell_labels.csv.gz`.
**Produces:** in `<STORAGE_DIR>/<run>/`: `reed_level2_classifier_cv.csv` (cross-validation results) and `reed_level2_classifier_coef.csv` (fitted coefficients). In `figures/<run>/03/`: the cross-validation curves and the test-set confusion matrix.

The model code (`MultinomialLogistic`, `WeightedGroupElasticNet`) is in `multinomial_group_lasso.py`, because skglm has no multinomial loss of its own. Run this notebook from inside `Reed_Explore/`, in the base conda environment.

In [ ]:
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmread
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)

from skglm.solvers import MultiTaskBCD

# The model code lives next to this notebook
from multinomial_group_lasso import MultinomialLogistic, WeightedGroupElasticNet

# Shared Python config (RNAseq_SingleCell/config.py): OUTPUT_DIR (the storage folder, from local_paths.env).
# The notebook runs from Reed_Explore/, so the config is one folder up.
sys.path.insert(0, os.path.abspath('..'))
from config import OUTPUT_DIR

# numba warns about non-contiguous arrays inside the solver, and scipy about a future change
# in mmread's return type; neither affects the results
warnings.filterwarnings('ignore', module='numba')
warnings.filterwarnings('ignore', category=DeprecationWarning)

## Settings

In [ ]:
cellranger_run_name = '90'
data_dir = os.path.join(OUTPUT_DIR, cellranger_run_name)       # where 02_Marker_Gene_Matrices.R saved its files
figure_outdir_cur = os.path.join('figures', cellranger_run_name, '03')
os.makedirs(figure_outdir_cur, exist_ok=True)

# The classes to predict. Every other level2 label becomes "Other".
KEEP_STATES = ['LASP1', 'LASP2', 'LASP3', 'LASP4', 'LASP5',
               'LHS1', 'LHS2', 'LHS3',
               'BMYO1', 'BMYO2']

TEST_SIZE = 0.3          # 70/30 train/test split
N_FOLDS = 5              # cross-validation folds (on the training set)
RANDOM_STATE = 0

# Penalty grid. l1_ratio = 1 is pure group lasso, 0 would be pure ridge.
# For each l1_ratio, alpha runs from alpha_max (every gene dropped) down to alpha_max * ALPHA_MIN_RATIO.
L1_RATIOS = [0.25, 0.5, 0.75, 1.0]
N_ALPHAS = 10
ALPHA_MIN_RATIO = 1e-3

# Solver settings
TOL = 1e-6
MAX_ITER = 100

# Optional: cap the number of cells per class before splitting, to make tuning faster.
# None uses every cell (~803,000 cells x ~3,000 genes, which is slow: 5 folds x the whole grid).
MAX_CELLS_PER_CLASS = None

## Load the cell state marker matrix and the labels

In [ ]:
prefix = os.path.join(data_dir, 'reed_state_marker_expr')

# 02 saved the matrix as genes x cells; the model needs cells x genes.
# CSC (compressed by column) is the sparse layout the solver expects.
X_all = sp.csc_matrix(mmread(prefix + '.mtx.gz').T)
genes = pd.read_csv(prefix + '.genes.txt.gz', header=None)[0].astype(str).to_numpy()
cells = pd.read_csv(prefix + '.cells.txt.gz', header=None)[0].astype(str).to_numpy()

labels = pd.read_csv(os.path.join(data_dir, 'reed_cell_labels.csv.gz'))

# The label file and the matrix columns must list the cells in the same order
assert X_all.shape == (len(cells), len(genes))
assert (labels['cell'].astype(str).to_numpy() == cells).all()

print(f'{X_all.shape[0]:,} cells x {X_all.shape[1]:,} marker genes')

In [ ]:
# Target: the level2 cell state for the epithelial states we care about, "Other" for everything else
y_all = np.where(labels['level2'].isin(KEEP_STATES), labels['level2'], 'Other')
classes = np.array(KEEP_STATES + ['Other'])      # fixed class order for coefficients and plots

pd.Series(y_all).value_counts().reindex(classes).to_frame('cells')

In [ ]:
# Optional subsampling (see MAX_CELLS_PER_CLASS above)
rng = np.random.default_rng(RANDOM_STATE)
if MAX_CELLS_PER_CLASS is not None:
    keep = np.concatenate([
        rng.choice(np.flatnonzero(y_all == c), size=min(MAX_CELLS_PER_CLASS, (y_all == c).sum()), replace=False)
        for c in classes
    ])
    keep.sort()
    X_all, y_all, cells = X_all[keep], y_all[keep], cells[keep]
    print(f'Subsampled to {len(keep):,} cells')

## Train/test split, scaling and intercept

The split is stratified, so each class keeps the same share in both sets. Genes are scaled to unit variance using the **training set only**, so nothing about the test set leaks into the model. They are not centred, which keeps the matrix sparse. The intercept column absorbs the offsets instead.

In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(X_all.shape[0]), test_size=TEST_SIZE, stratify=y_all, random_state=RANDOM_STATE)

y_train, y_test = y_all[idx_train], y_all[idx_test]

def add_intercept(X):
    """Put a column of ones in front of X (as column 0), in CSC format."""
    return sp.hstack([sp.csc_matrix(np.ones((X.shape[0], 1))), X], format='csc')

def one_hot(y):
    """One-hot encode labels in the fixed class order."""
    return (y[:, None] == classes[None, :]).astype(np.float64)

feature_names = np.concatenate([['(intercept)'], genes])

# Penalty weights: 0 for the intercept (not penalized), 1 for every gene
penalty_weights = np.r_[0., np.ones(len(genes))]

print(f'train: {len(idx_train):,} cells, test: {len(idx_test):,} cells')

## Model fitting helpers

In [ ]:
def fit_path(X, Y, alphas, l1_ratio):
    """Fit the model for each alpha, from largest to smallest, warm-starting each fit from the previous one.
    Returns a list of coefficient matrices W (n_features x n_classes), one per alpha."""
    datafit = MultinomialLogistic()
    solver = MultiTaskBCD(fit_intercept=False, tol=TOL, max_iter=MAX_ITER)
    W = np.zeros((X.shape[1], Y.shape[1]))
    path = []
    for alpha in alphas:
        penalty = WeightedGroupElasticNet(alpha, l1_ratio, penalty_weights)
        W, _, _ = solver.solve(X, Y, datafit, penalty, W.copy(), X @ W)
        path.append(W.copy())
    return path

def predict_proba(X, W):
    """Class probabilities (softmax of the scores), one column per class."""
    scores = X @ W
    scores -= scores.max(axis=1, keepdims=True)
    P = np.exp(scores)
    return P / P.sum(axis=1, keepdims=True)

def mean_log_loss(y, P):
    """Mean cross-entropy of true labels y under probabilities P (columns in the order of classes).
    sklearn's log_loss assumes alphabetical column order, which classes is not."""
    col = pd.Index(classes).get_indexer(y)
    return -np.mean(np.log(np.clip(P[np.arange(len(y)), col], 1e-15, None)))

def alpha_max(X, Y, l1_ratio):
    """Smallest alpha at which every gene is dropped (only the intercept is fitted).
    At that point the predicted probabilities are just the class frequencies."""
    P0 = np.tile(Y.mean(axis=0), (X.shape[0], 1))
    grad = X.T @ (P0 - Y) / X.shape[0]             # gradient at W = intercept only
    grad_norms = np.linalg.norm(np.asarray(grad), axis=1)[1:]   # skip the intercept row
    return grad_norms.max() / l1_ratio

## 5-fold cross-validation over `alpha` and `l1_ratio`

For each `l1_ratio`, the `alpha` grid is computed once from the full training set and shared by all folds. In each fold, the scaler is fitted on that fold's training part only. The score is the held-out **log-loss** (lower is better). Accuracy and balanced accuracy are recorded too.

In [ ]:
X_train_raw = X_all[idx_train]
Y_train = one_hot(y_train)

# Scale on the whole training set, only to set the alpha grids
X_train_for_grid = add_intercept(StandardScaler(with_mean=False).fit_transform(X_train_raw))
alpha_grids = {
    l1r: alpha_max(X_train_for_grid, Y_train, l1r) * np.geomspace(1, ALPHA_MIN_RATIO, N_ALPHAS)
    for l1r in L1_RATIOS
}

folds = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
t0 = time.time()
for fold, (tr, va) in enumerate(folds.split(X_train_raw, y_train)):
    scaler = StandardScaler(with_mean=False).fit(X_train_raw[tr])
    X_tr = add_intercept(scaler.transform(X_train_raw[tr]))
    X_va = add_intercept(scaler.transform(X_train_raw[va]))
    for l1r in L1_RATIOS:
        path = fit_path(X_tr, Y_train[tr], alpha_grids[l1r], l1r)
        for alpha, W in zip(alpha_grids[l1r], path):
            P = predict_proba(X_va, W)
            pred = classes[P.argmax(axis=1)]
            cv_rows.append(dict(
                fold=fold, l1_ratio=l1r, alpha=alpha,
                log_loss=mean_log_loss(y_train[va], P),
                accuracy=accuracy_score(y_train[va], pred),
                balanced_accuracy=balanced_accuracy_score(y_train[va], pred),
                n_genes=int((np.linalg.norm(W[1:], axis=1) > 0).sum()),
            ))
        print(f'fold {fold + 1}/{N_FOLDS}, l1_ratio {l1r}: done ({time.time() - t0:.0f} s)')

cv_results = pd.DataFrame(cv_rows)
cv_summary = (cv_results.groupby(['l1_ratio', 'alpha'])
              .agg(log_loss=('log_loss', 'mean'), log_loss_se=('log_loss', 'sem'),
                   accuracy=('accuracy', 'mean'), balanced_accuracy=('balanced_accuracy', 'mean'),
                   n_genes=('n_genes', 'mean'))
              .reset_index())
cv_summary.to_csv(os.path.join(data_dir, 'reed_level2_classifier_cv.csv'), index=False)

best = cv_summary.loc[cv_summary['log_loss'].idxmin()]
best_l1_ratio, best_alpha = best['l1_ratio'], best['alpha']
print(f"Best: l1_ratio = {best_l1_ratio}, alpha = {best_alpha:.3g} "
      f"(mean CV log-loss {best['log_loss']:.4f}, ~{best['n_genes']:.0f} genes)")

In [ ]:
# CV curves: mean held-out log-loss (+/- 1 standard error) against alpha, one line per l1_ratio
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for l1r, d in cv_summary.groupby('l1_ratio'):
    axes[0].errorbar(d['alpha'], d['log_loss'], yerr=d['log_loss_se'], marker='o', ms=3, capsize=2, label=f'l1_ratio = {l1r}')
    axes[1].plot(d['alpha'], d['n_genes'], marker='o', ms=3, label=f'l1_ratio = {l1r}')
axes[0].axvline(best_alpha, color='grey', ls='--', lw=1)
axes[0].set(xscale='log', xlabel='alpha', ylabel='Mean CV log-loss', title='Cross-validation')
axes[1].set(xscale='log', xlabel='alpha', ylabel='Genes kept (mean over folds)', title='Model size')
axes[0].legend()
fig.tight_layout()
fig.savefig(os.path.join(figure_outdir_cur, 'cv_curves.png'), dpi=200)

## Refit on the whole training set and evaluate on the test set

In [ ]:
scaler = StandardScaler(with_mean=False).fit(X_train_raw)
X_train = add_intercept(scaler.transform(X_train_raw))
X_test = add_intercept(scaler.transform(X_all[idx_test]))

# Warm-start along the grid down to the chosen alpha (faster than fitting it from scratch)
grid = alpha_grids[best_l1_ratio]
W_best = fit_path(X_train, Y_train, grid[grid >= best_alpha], best_l1_ratio)[-1]

P_test = predict_proba(X_test, W_best)
pred_test = classes[P_test.argmax(axis=1)]

print(f'Test log-loss:          {mean_log_loss(y_test, P_test):.4f}')
print(f'Test accuracy:          {accuracy_score(y_test, pred_test):.4f}')
print(f'Test balanced accuracy: {balanced_accuracy_score(y_test, pred_test):.4f}')
print()
print(classification_report(y_test, pred_test, labels=classes, zero_division=0))

In [ ]:
# Confusion matrix on the test set, as the fraction of each true class (rows sum to 1)
cm = confusion_matrix(y_test, pred_test, labels=classes, normalize='true')
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=classes, yticklabels=classes,
            vmin=0, vmax=1, cbar_kws={'label': 'Fraction of true class'}, ax=ax)
ax.set(xlabel='Predicted', ylabel='True', title='Test set confusion matrix')
fig.tight_layout()
fig.savefig(os.path.join(figure_outdir_cur, 'test_confusion_matrix.png'), dpi=200)

## Coefficients

`coef` has one row per feature (intercept first, then genes) and one column per class, on the scaled-gene scale. Genes dropped by the group lasso have all-zero rows. `importance` is the size of each gene's coefficient vector across classes.

In [ ]:
coef = pd.DataFrame(W_best, index=feature_names, columns=classes)
coef['importance'] = np.linalg.norm(W_best, axis=1)
coef.loc['(intercept)', 'importance'] = np.nan
coef.to_csv(os.path.join(data_dir, 'reed_level2_classifier_coef.csv'))

selected = coef.drop(index='(intercept)').query('importance > 0').sort_values('importance', ascending=False)
print(f'{len(selected):,} of {len(genes):,} genes kept')
selected.head(30)